# Data Contract & Governance: Content Decay Prediction System
### Engineering Reliability, Schema Assertions & Target Separation

> **Stakeholder Value:** In enterprise machine learning, over 80% of pipeline failures stem from silent data corruption, unexpected schema drift, or target leakage. This Data Contract acts as an operational SLA between data engineering and machine learning—guaranteeing that models receive clean, compliant data in production.

---

## 🛡️ Executive Summary & Purpose of the Data Contract

1. **Unit of Analysis (Grain):** Exactly 1 row = 1 unique content asset (`content_id`).
2. **Operational Guardrails:** Hard assertions on non-negative economics (`cpc >= 0`, `competition in [0, 1]`) and strict percentage bounds (`engagement_rate in [0, 100]`).
3. **Zero Target Leakage:** Strict mathematical isolation of all sub-interval 30-day windows (`*_last_30d`, `*_prev_30d`) and direct target components (`trend_pct`, `trend_direction`).
4. **Architectural Missingness:** Explicit missingness flags (`has_*`) ensure models distinguish between genuine zero metrics (e.g., zero clicks) and unmeasured metrics (e.g., non-text calculators).

```
Observation Window Architecture & Retrospective Overlap:
|--------------------- Trailing 90-Day Observation Window (Features) ---------------------|
|----- Days 61-90 Back -----|----- Days 31-60 Back (prev_30d) -----|----- Days 1-30 Back (last_30d) -----| [Export t_0]
                                                                     ^__________ Target Evaluation Window __|
```


In [1]:
import os
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
DATA_PATH = "../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns across {df['client_id'].nunique()} distinct clients.")


Loaded dataset: 30,000 rows x 44 columns across 32 distinct clients.


---

## 1. Unit of Analysis & Dataset Grain

### 1.1 Specification
- **Grain:** Exactly one record represents one unique published content asset (`content_id`).
- **Client Attribution:** Each content item is owned by exactly one client domain (`client_id`).
- **Population Scale:** 30,000 distinct content items across 32 enterprise client domains.

### 1.2 Automated Verification
We assert three non-negotiable data integrity checks:
1. `content_id` contains zero nulls and zero duplicate entries.
2. Total row count exactly equals `count(distinct content_id)`.
3. Every content asset maps strictly to one enterprise client portfolio (no split ownership).


In [2]:
# 1. Row count and uniqueness verification
total_rows = len(df)
unique_content_ids = df['content_id'].nunique()
duplicate_content_ids = total_rows - unique_content_ids
null_content_ids = df['content_id'].isnull().sum()

print(f"Total rows: {total_rows:,}")
print(f"Unique content_id: {unique_content_ids:,}")
print(f"Duplicate content_id: {duplicate_content_ids}")
print(f"Null content_id: {null_content_ids}")

# # Assertion test: Grain holds strictly
assert duplicate_content_ids == 0, "ERROR: Duplicate content_id detected in dataset grain!"
assert null_content_ids == 0, "ERROR: Null content_id detected in dataset grain!"

# 2. Relationship between content_id and client_id
clients_per_content = df.groupby('content_id')['client_id'].nunique()
assert (clients_per_content == 1).all(), "ERROR: Content items belong to multiple clients!"

unique_clients = df['client_id'].nunique()
print(f"Unique client_id count: {unique_clients} (Expected: 32)")
assert unique_clients == 32, f"Expected 32 clients, found {unique_clients}"
print("\nUnit of analysis strictly holds at 1 row = 1 unique content item.")

Total rows: 30,000
Unique content_id: 30,000
Duplicate content_id: 0
Null content_id: 0
Unique client_id count: 32 (Expected: 32)

Unit of analysis strictly holds at 1 row = 1 unique content item.


---

## 2. Time Horizons & Observation Windows

### 2.1 Specification & Business Context
All performance metrics represent an aggregated cross-sectional snapshot over an observation window ending at export time:
- **Content Maturity Floor:** `content_age_days` measures age since publication. All assets in this study satisfy `content_age_days >= 90` (mature content).
- **90-Day Observation Lookback:** 90-day search and engagement volume totals (`impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`).
- **Trajectory Windows for Trend Evaluation:**
  - `*_last_30d`: Recent 30-day performance window ($t_{-30} \rightarrow t_0$).
  - `*_prev_30d`: Prior 30-day baseline window ($t_{-60} \rightarrow t_{-31}$).
- **Temporal Alignment Rule:** Because `trend_direction` is derived by comparing `impressions_last_30d` to `impressions_prev_30d`, the 30-day window variables directly define the target outcome and must be protected.

### 2.2 Automated Verification
We assert:
1. `min(content_age_days) >= 90` and `min(impressions_90d) >= 1`.
2. Active engagement days are logically bounded within $[0, 90]$.
3. Lookback window consistency: `impressions_last_30d + impressions_prev_30d <= impressions_90d`.


In [3]:
# Verification of time window bounds
print(f"Content age range: [{df['content_age_days'].min()}, {df['content_age_days'].max()}] days")
print(f"Days with impressions range: [{df['days_with_impressions'].min()}, {df['days_with_impressions'].max()}] days")
print(f"Days with sessions range: [{df['days_with_sessions'].min()}, {df['days_with_sessions'].max()}] days")
print(f"Minimum impressions_90d: {df['impressions_90d'].min()}")

assert df['content_age_days'].min() >= 90, "ERROR: Content younger than 90 days found in mature slice!"
assert df['impressions_90d'].min() >= 1, "ERROR: Non-positive impressions_90d found!"
assert df['days_with_impressions'].max() <= 90, "ERROR: Days with impressions exceeds 90!"
assert df['days_with_sessions'].max() <= 90, "ERROR: Days with sessions exceeds 90!"

# Verification of window sub-intervals: last_30d + prev_30d should not exceed 90d
window_violations = (df['impressions_last_30d'] + df['impressions_prev_30d'] > df['impressions_90d']).sum()
print(f"Rows where (impressions_last_30d + impressions_prev_30d) > impressions_90d: {window_violations}")
assert window_violations == 0, "ERROR: 60-day subtotal exceeds 90-day total!"

print("\nObservation windows match specifications perfectly.")

Content age range: [90, 564] days
Days with impressions range: [1, 88] days
Days with sessions range: [1, 90] days
Minimum impressions_90d: 1
Rows where (impressions_last_30d + impressions_prev_30d) > impressions_90d: 0

Observation windows match specifications perfectly.


---

## 3. Column Partitioning & Zero-Leakage Governance

### 3.1 Partitioning Standards
To ensure data cleanliness and eliminate risk of target leakage, every column in the dataset is assigned to **exactly one** of four operational tiers:
1. **Candidate Features (32 columns):** Retrospective 90-day volume totals, derived user engagement rates, keyword economics, and content properties available prior to prediction time.
2. **Target Label & Mathematical Proxies (2 columns):** The ground truth outcome and variables mathematically derived from it. **Permanently excluded from feature matrices.**
3. **Context & Split Keys (2 columns):** Identifiers (`content_id`, `client_id`) used strictly for record tracking and grouped cross-validation. Never passed to model algorithms.
4. **Excluded Operational Columns (8 columns):** Intermediate 30-day sub-window metrics that comprise the target definition (leaked information), plus uninformative metadata flags.

### 3.2 Enterprise Column Registry (44 Columns Partitioned)

| Category | Columns | Count | Business & Technical Rationale |
|---|---|---|---|
| **Context** | `content_id`, `client_id` | 2 | Unique asset tracking key (`content_id`) and enterprise client group key (`client_id`) for out-of-domain cross-validation. |
| **Label / Target Proxy** | `trend_direction`, `trend_pct` | 2 | `is_declining_label` is defined as `(trend_direction == 'down')`. `trend_pct` is the mathematical derivation of `trend_direction`. |
| **Excluded (Target Leakage / Noise)** | `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`, `provider_used`, `model_used` | 8 | The 30-day window columns directly constitute the target label; including them would cause 100% artificial target leakage. AI provider flags contain high missingness and reflect production vendor choices rather than content decay mechanisms. |
| **Candidate Features** | All remaining 32 columns | 32 | Historical 90-day search visibility, user engagement ratios, keyword competition, and structural content attributes. |

### 3.3 Verification of Exhaustive Column Partitioning


In [4]:
# Define exhaustive column sets
CONTEXT_COLS = {'content_id', 'client_id'}

LABEL_COLS = {'trend_direction', 'trend_pct'}

EXCLUDED_COLS = {
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'provider_used', 'model_used'
}

FEATURE_COLS = set(df.columns) - CONTEXT_COLS - LABEL_COLS - EXCLUDED_COLS

print(f"Total columns in CSV: {len(df.columns)}")
print(f"Context columns ({len(CONTEXT_COLS)}): {sorted(list(CONTEXT_COLS))}")
print(f"Label columns ({len(LABEL_COLS)}): {sorted(list(LABEL_COLS))}")
print(f"Excluded columns ({len(EXCLUDED_COLS)}): {sorted(list(EXCLUDED_COLS))}")
print(f"Candidate feature columns ({len(FEATURE_COLS)}): {sorted(list(FEATURE_COLS))}")

# # Assertions: Mutual exclusivity and exhaustive coverage
assert len(CONTEXT_COLS & LABEL_COLS) == 0, "Overlap between Context and Label!"
assert len(CONTEXT_COLS & EXCLUDED_COLS) == 0, "Overlap between Context and Excluded!"
assert len(CONTEXT_COLS & FEATURE_COLS) == 0, "Overlap between Context and Feature!"
assert len(LABEL_COLS & EXCLUDED_COLS) == 0, "Overlap between Label and Excluded!"
assert len(LABEL_COLS & FEATURE_COLS) == 0, "Overlap between Label and Feature!"
assert len(EXCLUDED_COLS & FEATURE_COLS) == 0, "Overlap between Excluded and Feature!"

reconstructed_columns = CONTEXT_COLS | LABEL_COLS | EXCLUDED_COLS | FEATURE_COLS
assert reconstructed_columns == set(df.columns), "Partition does not exhaustively match CSV columns!"

print("\nAll 44 columns partitioned exhaustively with zero overlap.")

Total columns in CSV: 44
Context columns (2): ['client_id', 'content_id']
Label columns (2): ['trend_direction', 'trend_pct']
Excluded columns (8): ['clicks_last_30d', 'clicks_prev_30d', 'impressions_last_30d', 'impressions_prev_30d', 'model_used', 'provider_used', 'sessions_last_30d', 'sessions_prev_30d']
Candidate feature columns (32): ['age_tier', 'age_tier_order', 'ai_sessions_90d', 'ai_traffic_pct', 'avg_position', 'char_count', 'char_count_tier', 'clicks_90d', 'competition', 'competition_level', 'content_age_days', 'content_type', 'cpc', 'ctr', 'days_since_last_update', 'days_with_impressions', 'days_with_sessions', 'engaged_sessions_90d', 'engagement_rate', 'freshness_tier', 'impression_tier', 'impressions_90d', 'main_intent', 'pageviews_90d', 'position_tier', 'scroll_events_90d', 'scroll_rate', 'search_volume', 'sessions_90d', 'users_90d', 'word_count', 'word_count_tier']

All 44 columns partitioned exhaustively with zero overlap.


---

## 4. Missingness Governance & Platform Measurement Nuances

### 4.1 Understanding Systematic Data Gaps
Missing data in enterprise marketing and search platforms often carries structural meaning rather than random error:
1. **Keyword Context Gaps (`search_volume`, `cpc`, `competition`):** Missing systematically based on `content_type` (e.g. syndicated feed articles have 100% missing keyword data). Imputing zeros blindly would distort the feature space.
2. **Content Property Gaps (`word_count`, `char_count`):** Unpopulated for non-editorial formats (e.g., interactive landing pages, interactive product calculators, media hubs).
3. **Platform Measurement Quirks (Production Awareness):**
   - **Percentage Scaling:** Rates like `ctr`, `engagement_rate`, `scroll_rate`, and `ai_traffic_pct` are stored as percentages scaled by 100 (`0.76` represents `0.76%`, not `76%`).
   - **Zero Position Index:** `avg_position = 0` indicates missing ranking data, not a position 0 rank.
   - **Ratios Exceeding 100%:** Differences in measurement triggers between tracking systems can produce ratios slightly above 100%.

### 4.2 Automated Verification


In [5]:
# 1. Overall Missingness Profile
missing_summary = df.isnull().sum()
missing_pct = (missing_summary / len(df) * 100).round(2)
missing_table = pd.DataFrame({'Missing_Count': missing_summary, 'Missing_Pct': missing_pct})
missing_table = missing_table[missing_table['Missing_Count'] > 0].sort_values(by='Missing_Count', ascending=False)
print("Columns with Missing Values:")
print(missing_table)

# Null tolerance bounds assertion
assert df['impressions_90d'].isnull().sum() == 0, "Critical key impressions_90d cannot have nulls!"
assert df['content_age_days'].isnull().sum() == 0, "Critical key content_age_days cannot have nulls!"
assert missing_pct.max() < 75.0, "Unexpected null rate exceeding 75% tolerance!"

# 2. Structural Missingness across content_type
print("\nMissingness Rate by content_type:")
kw_missing = df.groupby('content_type')[['search_volume', 'cpc', 'word_count']].apply(lambda g: g.isnull().mean() * 100).round(1)
print(kw_missing)

assert kw_missing.loc['feedly article', 'search_volume'] == 100.0, "Expected feedly article to have 100% missing search_volume!"
print("\nKeyword missingness is strictly structural (100% on feedly articles).")

# 3. Explicit Allowable Range Assertions
assert (df['impressions_90d'] >= 1).all(), "All URLs must have impressions >= 1"
assert (df['content_age_days'] >= 90).all(), "All URLs must satisfy maturity floor >= 90 days"
assert (df['days_since_last_update'] >= 0).all(), "Days since last update cannot be negative"
assert (df['ctr'] >= 0).all(), "CTR cannot be negative"
assert (df['engagement_rate'] >= 0).all() and (df['engagement_rate'] <= 100).all(), "Engagement rate out of [0, 100] bounds!"
assert (df.dropna(subset=['cpc'])['cpc'] >= 0).all(), "CPC cannot be negative"
assert (df.dropna(subset=['competition'])['competition'].between(0, 1)).all(), "Competition score must be in [0, 1]"

# 4. Audit Measurement Quirks (Platform exceptions documented in Data Dictionary)
zero_position_count = (df['avg_position'] == 0).sum()
scroll_over_100 = (df['scroll_rate'] > 100).sum()
ai_traffic_over_100 = (df['ai_traffic_pct'] > 100).sum()

print(f"\navg_position == 0 count (no-data sentinel): {zero_position_count:,} ({zero_position_count/len(df):.2%})")
print(f"scroll_rate > 100 count: {scroll_over_100:,} ({scroll_over_100/len(df):.2%})")
print(f"ai_traffic_pct > 100 count: {ai_traffic_over_100:,} ({ai_traffic_over_100/len(df):.2%})")

assert zero_position_count == 1205, f"Expected 1,205 rows with avg_position == 0, found {zero_position_count}"
print("\nMeasurement quirks and allowable bounds confirmed matching data contract specifications.")

Columns with Missing Values:
                   Missing_Count  Missing_Pct
provider_used              21438        71.46
word_count_tier             7699        25.66
char_count                  7699        25.66
word_count                  7699        25.66
char_count_tier             7699        25.66
model_used                  5733        19.11
trend_pct                   3388        11.29
competition_level           2610         8.70
search_volume               2468         8.23
competition                 2468         8.23
cpc                         2468         8.23
main_intent                 2374         7.91
scroll_rate                  125         0.42

Missingness Rate by content_type:
                    search_volume    cpc  word_count
content_type                                        
comparison article            0.0    0.0         0.0
feedly article              100.0  100.0         0.0
keyword article               1.4    1.4        28.3

Keyword missingness is str

---

## 5. Target Label Formulation & Feature Store Output Specification

### 5.1 Business Target Definition: Content Decay (`is_declining_label`)
The primary business target identifies content suffering from material organic decay:
$$\text{is\_declining\_label} = \begin{cases} 1 & \text{if } \text{trend\_direction} = \text{'down'} \\ 0 & \text{otherwise} \end{cases}$$

Where `trend_direction` is triggered when an asset's search impressions decline by more than 20% over the trailing 30 days compared to the preceding 30 days:
$$\text{trend\_pct} = \frac{\text{impressions\_last\_30d} - \text{impressions\_prev\_30d}}{\text{impressions\_prev\_30d}} \times 100$$

### 5.2 Feature Engineering & Model Handoff Standards
The modeling pipeline receives a thoroughly validated dataset conforming to enterprise standards:
- Fully verified grain and grouped client partitions (`client_id`).
- Explicit boolean missingness indicators (`has_keyword_data`, `has_word_count`) alongside numerical handling.
- Standardized log-transforms (`log1p`) for heavy-tailed search volume metrics (`impressions_90d`, `clicks_90d`, `sessions_90d`).
- Absolute separation between feature matrices and target evaluation columns.

### 5.3 Automated Target Verification


In [6]:
# Construct and verify target distribution
df_contract = df.copy()
df_contract['is_declining_label'] = (df_contract['trend_direction'] == 'down').astype(int)

declining_count = df_contract['is_declining_label'].sum()
declining_rate = df_contract['is_declining_label'].mean()

print(f"Total items: {len(df_contract):,}")
print(f"Declining items (is_declining_label == 1): {declining_count:,} ({declining_rate:.2%})")
print(f"Non-declining items (is_declining_label == 0): {len(df_contract) - declining_count:,} ({1 - declining_rate:.2%})")

# Assert target base rate matches exact ground truth from data dictionary
assert declining_count == 16262, f"Expected 16,262 declining items, got {declining_count}"
assert round(declining_rate * 100, 2) == 54.21, f"Expected base rate 54.21%, got {declining_rate:.4f}"

print("\nBreakdown of trend_direction across target label:")
print(pd.crosstab(df_contract['trend_direction'], df_contract['is_declining_label'], margins=True))

print("\nData Contract checks passed completely. The dataset is certified for EDA & Signal Auditing.")

Total items: 30,000
Declining items (is_declining_label == 1): 16,262 (54.21%)
Non-declining items (is_declining_label == 0): 13,738 (45.79%)

Breakdown of trend_direction across target label:
is_declining_label      0      1    All
trend_direction                        
down                    0  16262  16262
flat                 1152      0   1152
new                  2236      0   2236
stable               5962      0   5962
up                   4388      0   4388
All                 13738  16262  30000

Data Contract checks passed completely. The dataset is certified for EDA & Signal Auditing.
